# 03 Feature Engineering

building engineered features, handling multicollinearity (correlation + VIF + PCA exploration), fitting a `ColumnTransformer` preprocessing pipeline exporting processed train/val/test splits.

In [1]:
from pathlib import Path
import json
import pickle
import sys

import pandas as pd

# NOTE: We import from src so notebook outputs match the production pipeline logic.
PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from oracle.features.champion_features import (
    apply_champion_encoders,
    fit_champion_encoders,
)
from oracle.features.pipeline import fit_transform_feature_splits
from oracle.features.player_features import add_player_features
from oracle.features.team_features import add_team_features

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

INTERIM_MERGED_PATH = INTERIM_DIR / "match_level_dataset.csv.gz"
TRAIN_PATH = PROCESSED_DIR / "train.csv.gz"
VAL_PATH = PROCESSED_DIR / "val.csv.gz"
TEST_PATH = PROCESSED_DIR / "test.csv.gz"

In [2]:
# NOTE: This notebook intentionally consumes curated train/val/test splits from the pipeline.
missing_inputs = [p for p in [TRAIN_PATH, VAL_PATH, TEST_PATH] if not p.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Missing processed split files. Run scripts/run_pipeline.py first. Missing: "
        + ", ".join(str(p) for p in missing_inputs)
    )

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)
merged_df = pd.read_csv(INTERIM_MERGED_PATH) if INTERIM_MERGED_PATH.exists() else None

{
    "train": train_df.shape,
    "val": val_df.shape,
    "test": test_df.shape,
    "target_train_rate": round(float(train_df["win"].mean()), 4),
    "merged_reference": (
        merged_df.shape
        if merged_df is not None
        else "missing: run scripts/run_pipeline.py"
    ),
}

{'train': (42, 138),
 'val': (6, 138),
 'test': (12, 138),
 'target_train_rate': 0.5,
 'merged_reference': (60, 138)}

In [3]:
train_enriched = add_team_features(add_player_features(train_df))
val_enriched = add_team_features(add_player_features(val_df))
test_enriched = add_team_features(add_player_features(test_df))

# NOTE: Target encoders are fit on train only, then reused on val/test to avoid leakage.
champion_artifacts = fit_champion_encoders(train_enriched, target_col="win")
train_enriched = apply_champion_encoders(train_enriched, champion_artifacts)
val_enriched = apply_champion_encoders(val_enriched, champion_artifacts)
test_enriched = apply_champion_encoders(test_enriched, champion_artifacts)

module_summary = {
    "champion_encoding_columns": sorted(champion_artifacts.mappings.keys()),
    "champion_encoding_prior": float(champion_artifacts.global_rate),
}
module_summary

{'champion_encoding_columns': [], 'champion_encoding_prior': 0.5}

In [4]:
# NOTE: ID/target/metadata columns are excluded by design from candidate engineered features.
reserved_cols = {
    "matchid",
    "teamid",
    "win",
    "duration",
    "gameid",
    "queueid",
    "seasonid",
    "creation",
}

candidate_features = [
    col
    for col in train_enriched.select_dtypes(include=["number"]).columns
    if col not in reserved_cols
]

# NOTE: Feature reduction order is correlation -> VIF, while PCA is tracked as a diagnostic metric.
(
    train_features_df,
    val_features_df,
    test_features_df,
    pipeline_summary,
    preprocessor,
) = fit_transform_feature_splits(
    train_enriched,
    val_enriched,
    test_enriched,
    feature_cols=candidate_features,
    target_col="win",
    id_cols=["matchid", "teamid"],
)

{
    "candidate_feature_count": pipeline_summary["candidate_feature_count"],
    "after_correlation": pipeline_summary["after_correlation"],
    "after_vif": pipeline_summary["after_vif"],
    "final_selected": pipeline_summary["final_selected"],
    "pca_components_95pct": pipeline_summary["pca_components_95pct"],
}

/home/amir/dev/lol-match-oracle/.venv/lib/python3.12/site-packages/sklearn/decomposition/_pca.py:646: RuntimeWarning: invalid value encountered in divide
  explained_variance_ratio_ = explained_variance_ / total_var


{'candidate_feature_count': 157,
 'after_correlation': 95,
 'after_vif': 13,
 'final_selected': 13,
 'pca_components_95pct': 1}

In [5]:
train_out = PROCESSED_DIR / "train_features.csv.gz"
val_out = PROCESSED_DIR / "val_features.csv.gz"
test_out = PROCESSED_DIR / "test_features.csv.gz"
preprocessor_out = PROCESSED_DIR / "feature_preprocessor.pkl"
summary_out = PROCESSED_DIR / "feature_engineering_summary.json"

train_features_df.to_csv(train_out, index=False, compression="gzip")
val_features_df.to_csv(val_out, index=False, compression="gzip")
test_features_df.to_csv(test_out, index=False, compression="gzip")

with preprocessor_out.open("wb") as handle:
    pickle.dump(preprocessor, handle)

feature_summary = {
    "scope": "post-game analysis with team-aggregated features",
    "dataset_granularity": "team-level",
    "inputs": {
        "merged_reference": str(INTERIM_MERGED_PATH),
        "train": str(TRAIN_PATH),
        "val": str(VAL_PATH),
        "test": str(TEST_PATH),
    },
    "outputs": {
        "train_features": str(train_out),
        "val_features": str(val_out),
        "test_features": str(test_out),
        "preprocessor": str(preprocessor_out),
    },
    "feature_counts": {
        "candidate": int(pipeline_summary["candidate_feature_count"]),
        "after_correlation": int(pipeline_summary["after_correlation"]),
        "after_vif": int(pipeline_summary["after_vif"]),
        "final_selected": int(pipeline_summary["final_selected"]),
    },
    "multicollinearity": {
        "dropped_high_correlation": pipeline_summary["dropped_high_correlation"],
        "dropped_high_vif": pipeline_summary["dropped_high_vif"],
        "pca_components_95pct": int(pipeline_summary["pca_components_95pct"]),
        "vif_top": pipeline_summary["vif_top"],
    },
    "transformations": {
        "robust_scaled_cols": pipeline_summary["robust_scaled_cols"],
        "standard_scaled_cols": pipeline_summary["standard_scaled_cols"],
    },
    "selected_features": pipeline_summary["selected_features"],
    "module_summary": module_summary,
    "rows": {
        "train": int(len(train_features_df)),
        "val": int(len(val_features_df)),
        "test": int(len(test_features_df)),
    },
}

# NOTE: Summary JSON is the compact record of feature choices and outputs for review.
with summary_out.open("w", encoding="utf-8") as handle:
    json.dump(feature_summary, handle, indent=2, sort_keys=True)
    handle.write("\n")

feature_summary

{'scope': 'post-game analysis with team-aggregated features',
 'dataset_granularity': 'team-level',
 'inputs': {'merged_reference': '/home/amir/dev/lol-match-oracle/data/interim/match_level_dataset.csv.gz',
  'train': '/home/amir/dev/lol-match-oracle/data/processed/train.csv.gz',
  'val': '/home/amir/dev/lol-match-oracle/data/processed/val.csv.gz',
  'test': '/home/amir/dev/lol-match-oracle/data/processed/test.csv.gz'},
 'outputs': {'train_features': '/home/amir/dev/lol-match-oracle/data/processed/train_features.csv.gz',
  'val_features': '/home/amir/dev/lol-match-oracle/data/processed/val_features.csv.gz',
  'test_features': '/home/amir/dev/lol-match-oracle/data/processed/test_features.csv.gz',
  'preprocessor': '/home/amir/dev/lol-match-oracle/data/processed/feature_preprocessor.pkl'},
 'feature_counts': {'candidate': 157,
  'after_correlation': 95,
  'after_vif': 13,
  'final_selected': 13},
 'multicollinearity': {'dropped_high_correlation': ['pentakills_sum',
   'goldspent_sum',
  

In [6]:
train_features_df.head()

,matchid,teamid,legendarykills_sum,timecc_sum,wardsbought_sum,legendarykills_mean,timecc_mean,wardsbought_mean,legendarykills_rate,timecc_rate,wardsbought_rate,participant_win_consistency,participant_win_inconsistent,cc_per_min,champion_signal_prior,win
0,10,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
1,10,200,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1
2,11,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
3,11,200,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1
4,12,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0


In [7]:
target_corr = (
    train_features_df.corr(numeric_only=True)["win"]
    .drop("win")
    .abs()
    .sort_values(ascending=False)
    .head(20)
)
target_corr

teamid                          0.047619
matchid                         0.000000
legendarykills_sum                   NaN
timecc_sum                           NaN
wardsbought_sum                      NaN
legendarykills_mean                  NaN
timecc_mean                          NaN
wardsbought_mean                     NaN
legendarykills_rate                  NaN
timecc_rate                          NaN
wardsbought_rate                     NaN
participant_win_consistency          NaN
participant_win_inconsistent         NaN
cc_per_min                           NaN
champion_signal_prior                NaN
Name: win, dtype: float64

In [8]:
{
    "saved_train_features": str(train_out),
    "saved_val_features": str(val_out),
    "saved_test_features": str(test_out),
    "saved_preprocessor": str(preprocessor_out),
    "saved_summary": str(summary_out),
}

{'saved_train_features': '/home/amir/dev/lol-match-oracle/data/processed/train_features.csv.gz',
 'saved_val_features': '/home/amir/dev/lol-match-oracle/data/processed/val_features.csv.gz',
 'saved_test_features': '/home/amir/dev/lol-match-oracle/data/processed/test_features.csv.gz',
 'saved_preprocessor': '/home/amir/dev/lol-match-oracle/data/processed/feature_preprocessor.pkl',
 'saved_summary': '/home/amir/dev/lol-match-oracle/data/processed/feature_engineering_summary.json'}